# H2 GuardedEvidence v3 Evaluation on Vertex AI (DeepSeek-V4.1-Flash-MaaS)

**Objective**: Run complete evaluation of H2 GuardedEvidence v3 on Vertex AI using model `deepseek-ai/deepseek-v4.1-flash-maas` on Devign (2,732 prompts) and ReVeal (2,274 prompts).

- **GCP Project**: `grace-research-2903` (location: `global`)
- **Key File**: `.gcp_key.json`
- **Evaluation Contract**: `evaluation_contract.yaml` (context_limit=8192, temp=0.0, seed=42)

In [ ]:
import os
import sys
import json
import time
import shutil
import zipfile
from pathlib import Path

# 1. Environment & Credentials Setup
REPO_ROOT = Path("..").resolve()
GCP_KEY = REPO_ROOT / ".gcp_key.json"
BUNDLE_DIR = Path(r"D:\tai lieu hoc tap\Đề tài nghiên cứu khoa học\papers source code\SHARE_H2_VERTEX_DEEPSEEK_V41_CONTEXT8192_20261022\SHARE_H2_VERTEX_DEEPSEEK_V41_CONTEXT8192_20261022")
PROJECT_ID = "grace-research-2903"

assert GCP_KEY.exists(), f"GCP key not found: {GCP_KEY}"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(GCP_KEY.resolve())
os.environ["CLOUD_ML_PROJECT_ID"] = PROJECT_ID
os.environ["GCP_PROJECT"] = PROJECT_ID
os.environ["VERTEX_LOCATION"] = "global"

print(f"[OK] Credentials configured: {GCP_KEY.name}")
print(f"[OK] Project: {os.environ['GCP_PROJECT']}")

In [ ]:
# 2. Add bundle to path and import runner
if str(BUNDLE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR.resolve()))

import runner
from runner import Replay
print("[OK] Successfully imported Replay from runner.py")

## 3. Step 1: Smoke Test (limit=1, allow_api=True)

In [ ]:
for ds in ["devign", "reveal"]:
    print(f"\n>>> Running Smoke Test for {ds}...")
    replay = Replay(BUNDLE_DIR, ds, PROJECT_ID)
    start_t = time.perf_counter()
    status = replay.run(limit=1, allow_api=True)
    elapsed = (time.perf_counter() - start_t) * 1000
    
    first_p = replay.prompts[0]
    first_res = replay.result(first_p)
    res_status = first_res.get("status")
    pred = first_res.get("prediction")
    print(f"Time: {elapsed:.1f}ms | Status: {res_status} | Parsed label: {pred}")
    assert res_status == "valid", f"Expected status == 'valid', got {res_status}"
    assert pred in [0, 1], f"Expected label in [0, 1], got {pred}"
    print(f"[PASSED] Smoke test for {ds} succeeded!")

## 4. Step 2: Full Evaluation (Devign 2,732 prompts & ReVeal 2,274 prompts)

In [ ]:
output_dir = REPO_ROOT / "output" / "h2_vertex_deepseek_v41"
output_dir.mkdir(parents=True, exist_ok=True)

runs = [
    {"name": "devign", "prompts": 2732},
    {"name": "reveal", "prompts": 2274}
]

eval_summaries = {}

for r in runs:
    ds = r["name"]
    print(f"\n{'='*50}")
    print(f"Running Evaluation for {ds.upper()} ({r['prompts']} prompts)")
    print(f"{'='*50}")
    
    replay = Replay(BUNDLE_DIR, ds, PROJECT_ID)
    print(f"Pre-run status: {replay.status()}")
    
    start_t = time.perf_counter()
    final_status = replay.run(limit=None, allow_api=True)
    total_time = time.perf_counter() - start_t
    print(f"Finished {ds} in {total_time:.2f}s: {final_status}")
    
    # Export outputs
    report = replay.export()
    print(f"Export status: {report.get('status')}")
    
    # Copy artifacts
    ds_dest = output_dir / ds
    ds_dest.mkdir(parents=True, exist_ok=True)
    for fname in ["predictions.jsonl", "raw_responses.jsonl", "evaluation.json", "run_manifest.json"]:
        src_f = replay.out / fname
        if src_f.exists():
            shutil.copy2(src_f, ds_dest / fname)
            
    eval_summaries[ds] = report

## 5. Step 3 & 4: Metric Comparison with Baselines (B2, B3) & Packaging

In [ ]:
baselines = {
    "B2_1shot": {"f1": 0.3614, "recall": 0.2778, "precision": 0.5172, "accuracy": 0.4700, "mcc": -0.0292},
    "B3_contrastive": {"f1": 0.5500, "recall": 0.6111, "precision": 0.5000, "accuracy": 0.4600, "mcc": -0.1118}
}

print("=" * 85)
print(f"{'Metric':<18} | {'B2 (1-shot)':<12} | {'B3 (Contrastive)':<16} | {'H2 Devign':<14} | {'H2 ReVeal':<14}")
print("-" * 85)

d_m = eval_summaries.get("devign", {}).get("metrics", {})
r_m = eval_summaries.get("reveal", {}).get("metrics", {})

for m_key, m_name in [("f1_score", "F1-Score"), ("recall", "Recall"), ("precision", "Precision"), ("accuracy", "Accuracy"), ("mcc", "MCC")]:
    v_b2 = baselines["B2_1shot"].get("f1" if m_key == "f1_score" else m_key, 0.0)
    v_b3 = baselines["B3_contrastive"].get("f1" if m_key == "f1_score" else m_key, 0.0)
    v_dev = d_m.get(m_key, d_m.get("f1", 0.0) if m_key == "f1_score" else 0.0)
    v_rev = r_m.get(m_key, r_m.get("f1", 0.0) if m_key == "f1_score" else 0.0)
    print(f"{m_name:<18} | {v_b2:<12.4f} | {v_b3:<16.4f} | {v_dev:<14.4f} | {v_rev:<14.4f}")
print("=" * 85)

# Package to zip
zip_dest = output_dir / "H2_vertex_results.zip"
with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(output_dir):
        for f in files:
            p = Path(root) / f
            if p == zip_dest:
                continue
            zf.write(p, arcname=str(p.relative_to(output_dir)))
print(f"\n[OK] Packaged results to: {zip_dest} ({zip_dest.stat().st_size} bytes)")